<a href="https://colab.research.google.com/github/tomaszwienke-lgtm/learning-git-task/blob/master/Zadanie_hiperparametry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files
uploaded = files.upload()
print("Przesłane pliki:", list(uploaded.keys()))

📘 Hiperparametry Pipeline’a – strojenie modeli (Diabetes)
Autor: [Tomasz Wienke]
Data: [03.05.2026]
Cel: Dla stworzonego Pipeline’a do klasyfikacji cukrzycy znaleźć optymalne hiperparametry za pomocą GridSearchCV. Zadanie kończy moduł o Pipeline'ach.

#1. Wstęp – czym są hiperparametry i dlaczego je stroimy?
Hiperparametry to ustawienia modelu (np. liczba drzew w lesie losowym, siła regularyzacji), które nie są uczone bezpośrednio z danych. Ich optymalny dobór ma kluczowe znaczenie dla skuteczności modelu.

Pipeline łączy preprocessing z modelem, dzięki czemu GridSearchCV może przeszukiwać nie tylko parametry modelu, ale również parametry preprocessingu (np. strategię uzupełniania braków).

W tym notebooku wykorzystamy zbiór Diabetes i dla stworzonego pipeline’u przeprowadzimy strojenie hiperparametrów.

2. Import bibliotek i konfiguracja

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

# Ustawienie stylu wykresów
sns.set_style("whitegrid")

# Zachowanie nazw kolumn po transformacjach
from sklearn import set_config
set_config(transform_output="pandas")

3. Wczytanie danych i podział

In [ ]:
df = pd.read_csv('diabetes.csv')
print("Rozmiar danych:", df.shape)
display(df.head())

X = df.drop('Diabetic', axis=1)
y = df['Diabetic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Trening:", X_train.shape, "Test:", X_test.shape)

#4. Budowa uniwersalnego preprocessora
Używamy make_column_selector, aby pipeline automatycznie rozpoznawał typy kolumn. Dzięki temu kod jest w pełni uniwersalny.

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipe, make_column_selector(dtype_include=['object', 'category']))
], remainder='drop')

print("Uniwersalny preprocessor gotowy.")

#5. Pipeline + GridSearchCV dla regresji logistycznej
Zaczniemy od prostszego modelu, aby pokazać działanie GridSearchCV.

In [ ]:
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

# Siatka hiperparametrów
param_grid_lr = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear', 'saga']
}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_lr.fit(X_train, y_train)

print("Najlepsze parametry LR:", grid_lr.best_params_)
print("Najlepszy AUC (walidacja):", grid_lr.best_score_)

6. Ocena najlepszego modelu regresji logistycznej

In [ ]:
best_lr = grid_lr.best_estimator_
y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("AUC ROC:", roc_auc_score(y_test, y_proba))

#7. Pipeline + GridSearchCV dla XGBoost (model, który zwyciężył wcześniej)
XGBoost to potężny algorytm boostingowy. Sprawdzimy, czy strojenie poprawi jego wyniki.

In [ ]:
pipe_xgb = Pipeline([
    ('prep', preprocessor),
    ('clf', XGBClassifier(eval_metric='logloss', random_state=42))
])

param_grid_xgb = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [3, 6, 9],
    'clf__learning_rate': [0.01, 0.1, 0.3]
}

grid_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)

print("Najlepsze parametry XGB:", grid_xgb.best_params_)
print("Najlepszy AUC (walidacja):", grid_xgb.best_score_)

8. Ocena najlepszego modelu XGBoost

In [ ]:
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("AUC ROC:", roc_auc_score(y_test, y_proba_xgb))

# Macierz pomyłek
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Macierz pomyłek - XGBoost (po strojeniu)')
plt.xlabel('Przewidywane')
plt.ylabel('Rzeczywiste')
plt.show()

9. Zapis najlepszego modelu

In [ ]:
joblib.dump(best_xgb, 'diabetes_best_pipeline.pkl')
print("Model zapisany jako 'diabetes_best_pipeline.pkl'")

## ✅ Podsumowanie i wnioski – Strojenie hiperparametrów Pipeline'a

### 🎯 Cel zadania
Zadanie miało na celu pokazanie, jak w praktyce wykorzystać `GridSearchCV` do znalezienia optymalnych hiperparametrów dla modeli pracujących wewnątrz Pipeline'a.

### 🧪 Przeprowadzone eksperymenty
1.  **Regresja logistyczna**: Przeszukano 20 kombinacji hiperparametrów (`C`, `penalty`, `solver`).
2.  **XGBoost**: Przeszukano 27 kombinacji hiperparametrów (`n_estimators`, `max_depth`, `learning_rate`).

### 📊 Kluczowe wyniki
| Model | Najlepsze hiperparametry | AUC (test) | Wnioski |
|---|---|---|---|
| **Regresja Logistyczna** | `C=0.1`, `penalty='l1'` | 0.860 | Strojenie nie poprawiło znacząco wyniku. Sugeruje to, że dane zawierają zależności nieliniowe, których model liniowy nie jest w stanie uchwycić. |
| **XGBoost** | `lr=0.3`, `depth=3`, `n=100` | **0.993** | Model osiągnął doskonałą skuteczność, a strojenie nieznacznie poprawiło już i tak bardzo wysoki wynik bazowy. |

### 🔧 Wnioski techniczne
- **Pipeline + GridSearchCV** to potężne, bezpieczne i oszczędzające czas połączenie. Pozwala na przeszukiwanie hiperparametrów zarówno modelu, jak i procesu przygotowania danych, bez ryzyka wycieku danych.
- Dzięki wykorzystaniu `make_column_selector`, nasz **kod preprocessora jest w pełni uniwersalny** i może być użyty na dowolnym zbiorze danych tabelarycznych.
- **Wytrenowany model** (zapisany jako `.pkl`) jest już specyficzny dla danego problemu i oczekuje tych samych nazw kolumn, na których był trenowany. To kluczowe rozróżnienie między uniwersalnością kodu a specyficznością wdrożonego modelu.

### 🏁 Podsumowanie końcowe modułu
Ten projekt udowadnia, że potrafimy nie tylko budować zaawansowane potoki przetwarzania danych, ale również w pełni wykorzystywać ich możliwości do automatycznego dostrajania modeli. Stworzony system jest solidnym fundamentem pod wdrożenie produkcyjne.